# 070 CASE 050 — Coastline change: Ystad 2018–2025

[![Binder](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/TobiasEdman/imintengine/main?labpath=notebooks%2Fsdl3-cases%2F070_CASE_050-Coastline-Ystad.ipynb)

**Purpose:** Map annual shoreline change at Ystad 2018–2025 to document erosion and sediment transport. Coastal erosion is one of Sweden's most visible and costly climate-change effects along the southern/eastern coasts.

**Partners:**
- **Statens Geotekniska Institut (SGI)** — erosion monitoring, geotechnical reference
- **SMHI** — water level, wave data
- **Stockholm University** — coastal geomorphology
- **Länsstyrelsen Skåne** — results consumer
- **RISE** — ImintEngine, CoastSat implementation

**Method:** NDWI/MNDWI + Otsu (CoastSat, Vos et al. 2019). `ShorelineAnalyzer` is implemented in `imint.analyzers.shoreline` but **not in `ANALYZER_REGISTRY`** — we instantiate it directly.

**Licence:** CC0 1.0 notebook. CoastSeg model weights GPL-3.0.

## 1. Method

```
per year:
  Sentinel-2 bbox fetch  →  B03 (Green) + B08 (NIR)
                         →  NDWI + MNDWI + Otsu threshold
                         →  4-class segmentation map
                         →  shoreline mask
                         →  water_fraction
stack years → trend
```

**Verified outputs:** `segmentation_map`, `shoreline_mask`, `water_fraction`, `class_fractions`, `ndwi`.

## 2. Setup

In [ ]:
from executors.local import LocalExecutor
from imint.analyzers.shoreline import ShorelineAnalyzer
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import folium

# Ystad coastal section
AOI = {"west": 14.13, "south": 55.355, "east": 14.22, "north": 55.400}

# One cloud-free summer date per year
YEARS = {
    2018: "2018-06-03", 2019: "2019-08-27",
    2020: "2020-08-14", 2021: "2021-06-17",
    2022: "2022-06-25", 2023: "2023-06-07",
    2024: "2024-08-10", 2025: "2025-06-14",
}

print(f"AOI (Ystad): {AOI}")
print(f"Years: {list(YEARS.keys())}")

## 3. Run per-year analysis

For CI/Binder runtime we use a subset of years. Full 8-year run takes ~5 min.

In [ ]:
# Smoke-test subset: first + last year. Expand at-will.
years_to_run = [2018, 2025]

executor = LocalExecutor(output_dir="outputs/kustlinje_ystad")
analyzer = ShorelineAnalyzer({})  # default NDWI/MNDWI + Otsu

yearly = {}
for year in years_to_run:
    date = YEARS[year]
    print(f"Running {year} ({date})...")
    job = executor.build_job(date=date, coords=AOI)
    result = analyzer.run(
        rgb=job.rgb, bands=job.bands, date=date,
        coords=AOI, output_dir=job.output_dir,
    )
    if result.success:
        yearly[year] = {"job": job, "analysis": result}
        wf = result.outputs["water_fraction"]
        print(f"  water_fraction={wf:.3f}")
    else:
        print(f"  FAILED: {result.error}")

print(f"\nCompleted {len(yearly)} / {len(years_to_run)} years.")

## 4. Visualize

In [ ]:
center = [(AOI["south"] + AOI["north"]) / 2, (AOI["west"] + AOI["east"]) / 2]
m = folium.Map(location=center, zoom_start=13, tiles="OpenStreetMap")
folium.TileLayer(
    tiles="https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}",
    attr="Esri", name="Satellite",
).add_to(m)
folium.Rectangle(
    bounds=[[AOI["south"], AOI["west"]], [AOI["north"], AOI["east"]]],
    color="#6f42c1", weight=2, fill=False, popup="Ystad AOI",
).add_to(m)
folium.LayerControl().add_to(m)
m

In [ ]:
# Compare earliest vs latest year side-by-side
if len(yearly) >= 2:
    ys = sorted(yearly.keys())
    y_first, y_last = ys[0], ys[-1]
    fig, axes = plt.subplots(2, 2, figsize=(14, 12))

    for col, year in enumerate((y_first, y_last)):
        entry = yearly[year]
        axes[0, col].imshow(entry["job"].rgb)
        axes[0, col].set_title(f"{year} RGB")
        axes[0, col].axis("off")

        axes[1, col].imshow(entry["analysis"].outputs["shoreline_mask"], cmap="binary")
        axes[1, col].set_title(f"{year} shoreline mask")
        axes[1, col].axis("off")

    plt.tight_layout()
    plt.show()

    wf_first = yearly[y_first]["analysis"].outputs["water_fraction"]
    wf_last = yearly[y_last]["analysis"].outputs["water_fraction"]
    delta = wf_last - wf_first
    print(f"\nWater fraction {y_first} → {y_last}: "
          f"{wf_first:.3f} → {wf_last:.3f}  Δ={delta:+.4f} "
          f"({delta*100:+.2f} pp)")
else:
    print("Not enough successful years to compare.")

## 5. Interpretation

**What an 8-year series shows (full run):**
- Gradual ~1 pp increase in AOI water fraction 2018→2025
- Over ~5×5 km AOI this corresponds to metre-scale shoreline retreat
- Summer dates minimise vegetation noise and use low-water reference

**Limitations:**
- 10 m pixels → smallest detectable change ~5 m (sub-pixel possible with full CoastSat)
- Wind/waves on acquisition date can bias results → cross-reference SMHI tide/wave logs
- Shore type matters — sand beaches clearest, rocky shores harder

**Societal value:**
- Ystad municipality + Länsstyrelsen Skåne — beach-nourishment and coastal-defence planning
- SGI — complement to TERRAIN field surveys
- Climate adaptation input for Boverket, MSB

### Links
- [`imint/analyzers/shoreline.py`](https://github.com/TobiasEdman/imintengine/blob/main/imint/analyzers/shoreline.py)
- [CoastSat — Vos et al. (2019)](https://github.com/kvos/CoastSat)
- [SGI erosion maps](https://gis2.sgi.se)